# Installing Net2Brain

<img src="workshops/data/Net2Brain_Logo.png" width="25%" />

In [ ]:
#!pip install -U git+https://github.com/cvai-roig-lab/Net2Brain

# Step 2: Creating Representational Dissimilarity Matrices (RDMs) using `RDM Creator`

After completing Step 1, where the Feature Extractor extracts features from the specified layers, we can now proceed to Step 2, which involves calculating Representational Dissimilarity Matrices (RDMs) using the RDM Creator's built-in functionality.

We first initialize the RDM Creator class, which contains the function to generate RDMs. The RDM Creator class requires the following parameters:
- **device**: The device on which the RDMs will be generated. This can be 'cpu' or 'cuda'.
- **verbose**: Whether to print the progress of the RDM generation process.

To generate RDMs, the RDM Creator function requires:

- **feature_path**: The path to the .npz files containing the layer features for each image. These files should have a [Batch x Channel x Height x Width] format.
- **save_path** (optional): The location where the generated RDMs will be saved.
- **save_format** (optional): The format in which the RDMs will be saved. This can be 'npz' or 'pt'.
- **distance** (optional): The distance function to generate the RDMs. Default is the correlation distance.
- **standardize_on_dim** (optional): The dimension along which to standardize the features. Default is None.
- **chunk_size** (optional): The number of images to be processed at a time. This is useful when the number of images is too large to fit in memory. Default is all images.
- **multi_timepoint_rdms** (optional): Whether to create separate RDM per timepoint (for video 
models) and at which level. Defaults to None (single RDM from flattened features). If 'clip', 
this creates one RDM per video subclip and flattens the rest of the dimensions, while if 
'all_timepoints' it creates one RDM per video subclip and per timepoint.
- **dim_reduction** (optional): Same as in feature extraction. This provides a second point in the pipeline to apply dimensionality reduction (at the feature loading) and it is only possible when features are saved in separate and not consolidated format. Defaults to None.
- **n_samples_estim**: Same as in feature extraction.
- **n_components** (optional): Same as in feature extraction.
- **srp_before_pca** (optional): Same as in feature extraction.
- **max_dim_allowed** (optional): Same as in feature extraction.
- **pooling** (optional): Pooling method for variable-length features. Options: ``'mean'`` (average across sequence), ``'max'`` (maximum values across sequence), ``'first'`` (use first token/position, e.g., [CLS] token), ``'last'`` (use final token/position). Required when processing transformer/LLM features with different sequence lengths. Defaults to ``None`` for fixed-length features.


The function will then:

- **Output**: Create an RDM with the shape (#Images, #Images) for every specified layer.


In this example, we'll be using the AlexNet features generated earlier in the tutorial. The function can be called as follows:

In [ ]:
from net2brain.rdm_creation import RDMCreator
import shutil
import os

save_path = "PATH"
feat_path = "PATH"

if os.path.exists(save_path):
    shutil.rmtree(save_path)
    


#feats must already be: (num_images, feature_dim)
# Call the Class with the path to the features
creator = RDMCreator(verbose=True, device='cpu') 
save_path = creator.create_rdms(feature_path=feat_path, save_path=save_path, save_format='npz') 


#feat_path = "ResNet50_Feat"
#save_path = "ResNet50_RDM"


# Call the Class with the path to the feautures
#creator = RDMCreator(device='cpu', verbose=True)  
#creator.create_rdms(feature_path=feat_path, save_path=save_path, save_format='npz')  # This will create and save the RDMs at the specified save path



# code to fix nsd formating and make brain rdms  
befor: 8 subject (folders) -> 37 rois RDMs (.npz) -> ( img -> voxels) (379756,)  
after: 35 shared rois (.npz) -> 8 subjects (arrays) -> (img -> voxels) (872, 872)

In [ ]:
import numpy as np
import os
from scipy.spatial.distance import squareform

# =========================
# 1. Get shared ROIs
# =========================

def get_shared_rois_nsd(paths):
    """
    Retrieves ROI names that exist for all 8 subjects
    """
    # start with subj01
    sub01_path = paths["subj01_rdms"]
    file_list = os.listdir(sub01_path)

    roi_names = []
    for filename in file_list:
        roi_names.append(filename.split("_subj01")[0])

    # intersect with other subjects
    for subj in range(2, 9):
        subj_key = f"subj0{subj}_rdms"
        subj_path = paths[subj_key]

        file_list = os.listdir(subj_path)

        subj_roi_names = []
        for filename in file_list:
            subj_roi_names.append(filename.split(f"_subj0{subj}")[0])

        roi_names = list(set(roi_names) & set(subj_roi_names))

    return roi_names


roi_names = get_shared_rois_nsd(paths_NSD_shared)

print(f"Found {len(roi_names)} shared ROIs")


# =========================
# 2. Create brain RDMs
# =========================

data_path = paths_NSD_shared["root"]

save_dir = os.path.join(data_path, "brain_data")
os.makedirs(save_dir, exist_ok=True)

for roi in roi_names:
    print(f"Processing ROI: {roi}")

    # (subjects, images, images)
    roi_rdm = np.zeros((8, 872, 872))

    for subj in range(1, 9):
        subj_key = f"subj0{subj}_rdms"
        subj_path = paths_NSD_shared[subj_key]

        roi_filename = f"{roi}_subj0{subj}.npy"

        file_path = os.path.join(subj_path, roi_filename)

        # load flattened RDM
        subj_roi = np.load(file_path)

        # convert to full matrix
        subj_roi = squareform(subj_roi)

        # store
        roi_rdm[subj - 1, :, :] = subj_roi

    # save
    save_name = f"{roi}_RDMs.npz"

    np.savez(os.path.join(save_dir, save_name), roi_rdm)

print("✅ Brain RDMs created successfully!")


In [ ]:
#testing
import numpy as np
import os

orig = np.load(os.path.join(
    paths_NSD_shared["subj01_rois"],
    "V1d_lh_subj01.npy"
))

print(orig.shape)


orig = np.load(os.path.join(
    paths_NSD_shared["subj02_rdms"],
    "V1d_lh_subj02.npy"
))

print(orig.shape)

orig = np.load(os.path.join(
    paths_NSD_shared["subj02_fmri"],
    "lh_fmri.npy"
))

print(orig.shape)

# Combine LH + RH 

In [ ]:
## test with one 

In [ ]:
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

#paths_bonner = DatasetBonnerPNAS2017.load_dataset()
#pprint(paths_bonner)

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

experimenting with 1 subject

In [ ]:
import numpy as np
import os

lh = np.load(os.path.join(
    paths_NSD_shared["subj01_rois"],
    "EBA_lh_subj01.npy"
))

rh = np.load(os.path.join(
    paths_NSD_shared["subj01_rois"],
    "EBA_rh_subj01.npy"
))

print("LH shape:", lh.shape)
print("RH shape:", rh.shape)

print("First image LH:", lh[0][:5])
print("First image RH:", rh[0][:5])

In [ ]:
combined = np.concatenate([lh, rh], axis=1)

print(combined.shape)

In [ ]:
import numpy as np
import os

save_dir = "combined_roi_features"
os.makedirs(save_dir, exist_ok=True)

np.savez(
    os.path.join(save_dir, "EBA_combined_subj01.npz"),
    arr_0=combined
)

In [ ]:
test = np.load("combined_roi_features/EBA_combined_subj01.npz")

print(test.files)
print(test["arr_0"].shape)

make 872 files, each containing rois of 1 img. that's what rdm creator expects

In [ ]:
import os
import numpy as np

save_dir = "combined_roi_features_correct"
os.makedirs(save_dir, exist_ok=True)

for i in range(combined.shape[0]):  # 872 images
    
    single_image_feat = combined[i]   # shape (6237,)

    np.savez(
        os.path.join(save_dir, f"{i:04d}.npz"),
        arr_0=single_image_feat
    )

print("done")

In [ ]:
creator = RDMCreator(verbose=True, device='cpu')

creator.create_rdms(
    feature_path="combined_roi_features_correct",
    save_path="combined_roi_rdms_correct",
    save_format='npz'
)

In [ ]:
import os
import numpy as np

files = os.listdir("combined_roi_rdms_correct")
print(files)

test = np.load(os.path.join(
    "combined_roi_rdms_correct",
    files[0]
))

print(test.files)

for k in test.files:
    print(k, np.array(test[k]).shape)

In [ ]:
# Convert the brain RDM back to square form.
from scipy.spatial.distance import squareform
import numpy as np
import os

# load condensed brain RDM
f = "combined_roi_rdms_correct/RDM_arr_0.npz"

data = np.load(f, allow_pickle=True)

rdm_condensed = data["rdm"]

# convert to square
rdm_square = squareform(rdm_condensed)

print(rdm_square.shape)

In [ ]:
import numpy as np
import os

# load square RDM
data = np.load(
    "brain_rdms_square/RDM_arr_0_square.npz",
    allow_pickle=True
)

rdm = data["rdm"]

print("before:", rdm.shape)

# add subject dimension
rdm = rdm[np.newaxis, :, :]

print("after:", rdm.shape)

# create final folder
os.makedirs("brain_rdms_final", exist_ok=True)

# save final RSA-compatible file
np.savez(
    "brain_rdms_final/RDM_arr_0_final.npz",
    rdm=rdm
)

print("saved")

In [ ]:
from scipy.spatial.distance import squareform
import numpy as np
import os

# load condensed RDM
data = np.load(
    "combined_roi_rdms_correct/RDM_arr_0.npz",
    allow_pickle=True
)

# get condensed vector
rdm_condensed = data["rdm"]

# convert to square matrix
rdm_square = squareform(rdm_condensed)

print(rdm_square.shape)   # should be (872, 872)

# create new folder
os.makedirs("brain_rdms_square", exist_ok=True)

# save square RDM
np.savez(
    "brain_rdms_square/RDM_arr_0_square.npz",
    rdm=rdm_square
)

print("done")

## combine all rois

In [ ]:
import os
import numpy as np
from scipy.spatial.distance import squareform
from net2brain.rdm_creation import RDMCreator

# =========================
# shared ROI names
# =========================

import os

valid_rois = []

subj01_files = os.listdir(paths_NSD_shared["subj01_rois"])

candidate_rois = []

for f in subj01_files:
    if "_lh_" in f:
        candidate_rois.append(f.split("_lh_")[0])

# check all subjects
for roi in candidate_rois:

    valid = True

    for subj in range(1, 9):

        subj_str = f"subj0{subj}"

        lh_path = os.path.join(
            paths_NSD_shared[f"{subj_str}_rois"],
            f"{roi}_lh_{subj_str}.npy"
        )

        rh_path = os.path.join(
            paths_NSD_shared[f"{subj_str}_rois"],
            f"{roi}_rh_{subj_str}.npy"
        )

        if not os.path.exists(lh_path) or not os.path.exists(rh_path):
            valid = False
            break

    if valid:
        valid_rois.append(roi)

print(valid_rois)
print("TOTAL:", len(valid_rois))

# =========================
# output folder
# =========================

os.makedirs("final_brain_rdms", exist_ok=True)

rdm_creator = RDMCreator(verbose=False, device='cpu')

# =========================
# process each ROI
# =========================

for roi in valid_rois:

    print(f"\nProcessing ROI: {roi}")

    all_subject_rdms = []

    # -------------------------
    # process each subject
    # -------------------------

    for subj in range(1, 9):

        subj_str = f"subj0{subj}"

        # load lh/rh
        lh = np.load(os.path.join(
            paths_NSD_shared[f"{subj_str}_rois"],
            f"{roi}_lh_{subj_str}.npy"
        ))

        rh = np.load(os.path.join(
            paths_NSD_shared[f"{subj_str}_rois"],
            f"{roi}_rh_{subj_str}.npy"
        ))

        # combine hemispheres
        combined = np.concatenate([lh, rh], axis=1)

        # -------------------------
        # temporary feature folder
        # -------------------------

        temp_feat_dir = os.path.join(
            "temp_features",
            roi,
            subj_str
        )

        os.makedirs(temp_feat_dir, exist_ok=True)

        # save one file per image
        for i in range(combined.shape[0]):

            np.savez(
                os.path.join(temp_feat_dir, f"{i:04d}.npz"),
                arr_0=combined[i]
            )

        # -------------------------
        # create RDM
        # -------------------------

        temp_rdm_dir = os.path.join(
            "temp_rdms",
            roi,
            subj_str
        )

        rdm_creator.create_rdms(
            feature_path=temp_feat_dir,
            save_path=temp_rdm_dir,
            save_format="npz"
        )

        # -------------------------
        # load condensed RDM
        # -------------------------

        data = np.load(
            os.path.join(temp_rdm_dir, "RDM_arr_0.npz"),
            allow_pickle=True
        )

        rdm_condensed = data["rdm"]

        # convert to square
        rdm_square = squareform(rdm_condensed)

        all_subject_rdms.append(rdm_square)

    # =========================
    # stack all subjects
    # =========================

    all_subject_rdms = np.stack(all_subject_rdms, axis=0)

    print("Final shape:", all_subject_rdms.shape)

    # =========================
    # save final brain RDM
    # =========================

    np.savez(
        os.path.join(
            "final_brain_rdms",
            f"{roi}_combined_RDMs.npz"
        ),
        rdm=all_subject_rdms
    )

print("\nDONE")

method 2: rdm avg. Load both .npz files → average them → save a new .npz

In [ ]:
"""
import numpy as np
import os

brain_path = "PATH"
save_path  = "PATH"

os.makedirs(save_path, exist_ok=True)

files = os.listdir(brain_path)

# find unique ROI bases
roi_bases = set(f.replace("_lh_RDMs.npz", "")
                  .replace("_rh_RDMs.npz", "")
                for f in files)

for roi in roi_bases:
    lh_file = os.path.join(brain_path, f"{roi}_lh_RDMs.npz")
    rh_file = os.path.join(brain_path, f"{roi}_rh_RDMs.npz")
    
    # check both exist
    if not (os.path.exists(lh_file) and os.path.exists(rh_file)):
        continue
    
    lh = np.load(lh_file)["arr_0"]   # (8, 872, 872)
    rh = np.load(rh_file)["arr_0"]
    
    combined = (lh + rh) / 2
    
    np.savez(os.path.join(save_path, f"{roi}_RDMs.npz"), combined)

print(" Combined brain RDMs created")
"""

In [ ]:
import os

print(os.listdir(brain_path)[:5])

#transforming files from npy to npz s.t. rdm creator can work

In [ ]:

import os
import numpy as np

input_folder = paths_NSD_shared["subj01_rois"]
output_folder = "brain_features_npz"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".npy"):
        data = np.load(os.path.join(input_folder, file))
        
        # IMPORTANT: do NOT reshape, do NOT modify
        print(file, data.shape)  # should be (872, X)

        np.savez(
            os.path.join(output_folder, file.replace(".npy", ".npz")),
            data  # ← correct (creates arr_0 internally)
        )


test = np.load("brain_features_npz/V1d_lh_subj01.npz")
print(test.files)        # should be ['arr_0']
print(test['arr_0'].shape)

In [ ]:
# creating rdms out of brain data
brain_rdm_save = "Brain_RDM"

creator = RDMCreator(verbose=True, device='cpu')

creator.create_rdms(
    feature_path="brain_features_single", 
    save_path="Brain_RDM_V1",
    save_format='npz'
)

In [ ]:
#testing
test = np.load("Brain_RDM_V1/RDM_arr_0.npz")
print(test.files)
print(test[test.files[0]].shape)

### Adding dimensionality reduction at the loading stage
If you wish you can also reduce the dimensionality of the loaded feautures using:

- `srp` (Sparse Random Projection)
- `pca` (Principal Component Analysis)
   - Can apply SRP before PCA for very high-dimensional data with `srp_before_pca=True`

This is only possible when the features are saved in separate format. If you want to save the features in 
consolidated format, and apply dimensionality reduction, it can only be done at the feature extraction stage and not 
at the loading stage.

In [ ]:
save_path = creator.create_rdms(feature_path=feat_path, save_path=save_path, save_format='npz', dim_reduction='srp') 

## Distance functions and normalization

The default distance function is the correlation distance. To use a different distance function, we can specify the distance function in the **distance** parameter. The available distance functions can been seen by calling the **distance_functions** method of the RDMCreator. We created synonyms for the distance functions to make it easier to use them (i.e. l2 == euclidean). The available distance functions are:

In [ ]:
creator.distance_functions()

You can also use custom distance functions by passing a function to the **distance** parameter. The function should take one argument `x` of shape `(N, D)`, which represents the features (dimension `D`) of the `N` stimuli and return a pairwise distance matrix of shape `(N, N)`. For example, we can use the cosine distance function as follows:

In [ ]:
import torch.nn.functional as F

def custom_cosine(x):
    x_norm = F.normalize(x, p=2, dim=1)
    return 1 - (x_norm @ x_norm.T)

creator.create_rdms(feature_path=feat_path, save_path='AlexNet_RDM_custom', save_format='npz', distance=custom_cosine)

## Feature standardization 

The input feature are not standardized by default. To standardize the input features, we can specify the dimension along which to standardize the features using the **standardize_on_dim** parameter. For an input of shape (num_stimuli, num_features) three options are available:
- `standardize_on_dim=None`: No standardization is applied.
- `standardize_on_dim=0`: Standardize along the first dimension (num_stimuli).
- `standardize_on_dim=1`: Standardize along the second dimension (num_features).

## Accessing and Visualizing RDMs

The `LayerRDM` class provides a convenient way to access and visualize the Representational Dissimilarity Matrix (RDM) for a single layer of data. To optimize memory usage, only the upper triangular part of the RDM is stored. The `LayerRDM` class efficiently manages the conversion between this upper triangular format and the full square RDM, and also facilitates visualization.

In [ ]:
from net2brain.rdm_creation import LayerRDM

rdm = LayerRDM.from_file("AlexNet_RDM/RDM_features_0.npz")
print("Class:", rdm)
rdm.plot(indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

The RDM data can be accessed through the `data` attribute, which returns the full square RDM:

In [ ]:
rdm.data

## RDMs of arbitrary data

The functional API provides a straightforward way to calculate the RDM for any arbitrary data. The `dist` function from ` net2brain.rdm.dist_utils` is designed to handle this process efficiently, supporting both individual feature matrices and batches of feature matrices. Key features of this function include:

   - **Sanity Checks**: Ensures that inputs are valid and correctly formatted.
   - **Data Types and Devices**: Automatically handles appropriate data types and computational devices.
   - **Chunking**: Efficiently processes large datasets by breaking them into smaller chunks.

For users who only need to calculate the distance, the raw function can be accessed directly from `net2brain.rdm.dist`.

In [ ]:
import torch
from net2brain.rdm.dist_utils import dist
from net2brain.rdm.dist import correlation

# Generate random example data
N, D = 10, 100
feature = torch.rand(N, D)

# Calculate RDM
raw_rdm = dist(feature, metric="correlation", chunk_size=None, verbose=True, device='cpu')
# or
raw_rdm = correlation(feature)

print("Shape of RDM (N x N):", raw_rdm.shape)

# Temporally resolved RDM construction for video features

For video models, it is often useful to create temporally resolved RDMs that capture the representational dissimilarities at different time points. The `multi_timepoint_rdms` parameter in the `create_rdms` function allows for this functionality. 

When set to `'clip'`, the function creates one RDM per video subclip, flattening the other 
dimensions. Make sure that your features were extracted with the keyword argument `agg_frames` 
either set to None or 'within_clips' during feature extraction for this to work properly.

When set to `'all_timepoints'`, it creates one RDM per video subclip and per timepoint. Make sure
that your features were extracted with the keyword argument `agg_frames` either set to None or 
'across_clips' during feature extraction for this to work properly.

In [ ]:
from net2brain.rdm_creation import RDMCreator

feat_path = 'VideoModel_TimeResolvedFeat'
save_path = 'VideoModel_TimeResolvedRDM'

creator = RDMCreator(device='cpu', verbose=True)  
creator.create_rdms(feature_path=feat_path, save_path=save_path, save_format='npz', multi_timepoint_rdms='all_timepoints')

## RDMs with Variable-Length Features
When working with features that have variable sequence lengths (such as text or time-series data), pooling is required to create uniform feature representations before calculating RDMs:

In [ ]:
import numpy as np
from pathlib import Path
from net2brain.rdm_creation import RDMCreator
from net2brain.rdm.rdm import LayerRDM
import shutil

# Create temporary dummy data
temp_dir = Path("./temp_features")
temp_dir.mkdir(exist_ok=True)

# Variable-length features (like BERT tokens)
for i, seq_len in enumerate([8, 12, 6]):
    data = {}
    for layer in range(2):
        data[f"encoder.layer.{layer}"] = np.random.randn(seq_len, 768).astype(np.float32)
    np.savez(f"temp_features/stimulus_{i:03d}.npz", **data)

# Create RDMs with pooling
creator = RDMCreator(verbose=True, device='cpu')
save_path = creator.create_rdms(
    feature_path="temp_features", 
    pooling="mean",
    save_format='npz'
)

# Plot the first RDM
rdm_files = list(save_path.glob("*.npz"))
rdm = LayerRDM.from_file(rdm_files[0])
rdm.plot()

# Clean up temporary files
shutil.rmtree("temp_features", ignore_errors=True)
shutil.rmtree(save_path, ignore_errors=True)